# init
used to query DB for spectra and labels

In [1]:
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

from pptoolbox.connectors import BaseEFSConnector, PFDBConnector

from datetime import datetime

In [2]:
load_dotenv(find_dotenv())

PF_SQL_PASSWORD = os.environ.get("UAT_SQL_PASSWORD", None)  # UAT_SQL_PASSWORD
PF_KEY_PATH = os.environ.get("UAT_SERVER_KEYPATH", None)  # UAT_SERVER_KEYPATH
PF_EFS_URL = os.environ.get("PF_EFS_URL", None)  # PF_EFS_URL
PF_CA_CERT = os.environ.get("UAT_CA_CERT", None)  # UAT_CA_CERT

print(PF_SQL_PASSWORD,PF_KEY_PATH,PF_EFS_URL,PF_CA_CERT)

password /Users/RyanSoh/.ssh/NextPlatform-UAT.pem http://54.254.245.207:5014 /Users/RyanSoh/.ssh/global-bundle.pem


In [3]:
raw_data_path = Path ("../data/raw")
output_folder = raw_data_path / "v1"
output_folder.mkdir(parents=True, exist_ok=True)

# by product_type

In [9]:
company_id = 1344
product_type = 7136
today_date = datetime.now().strftime("%y%m%d")

print (today_date)

260602


## label

In [ ]:
BASE_QUERY = f"""
WITH all_properties AS (
    -- Combine numerical and option properties first
    SELECT 
        lnpv.lot_id, 
        p.name AS property_name, 
        lnpv.value AS property_value
    FROM lot_numerical_property_value lnpv
    INNER JOIN numerical_property np ON np.id = lnpv.numerical_property_id
    INNER JOIN property p ON np.property_id = p.id

    UNION ALL

    SELECT 
        lopv.lot_id, 
        p.name AS property_name, 
        lopv.value AS property_value
    FROM lot_option_property_value lopv
    INNER JOIN option_property op ON op.id = lopv.option_property_id
    INNER JOIN property p ON op.property_id = p.id
)

SELECT 
    l.id AS lot_id, 
    l.name AS lot_name, 
    ap.property_name,
    ap.property_value,
    l.company_id,
    l.product_type_id,
    pt.name AS product_name
FROM lot l
-- LEFT JOIN ensures all lots are kept even if no match exists in 'all_properties'
LEFT JOIN all_properties ap ON l.id = ap.lot_id
INNER JOIN product_type pt ON pt.id = l.product_type_id
WHERE l.company_id = {company_id}
  AND l.product_type_id = {product_type}
  AND l.name NOT LIKE '%DELETED%'
ORDER BY l.id;
"""

In [ ]:
# BASE_QUERY = f"""
# SELECT lnpv.lot_id, lot.name as lot_name, 
# 	   property.name as property_name,
#        lnpv.value as property_value,
#        lot.company_id,
#        lot.product_type_id,
#        product_type.name as product_name
# FROM lot_numerical_property_value lnpv
# INNER JOIN lot on lot.id = lnpv.lot_id
# INNER JOIN numerical_property on numerical_property.id = lnpv.numerical_property_id
# INNER JOIN property on numerical_property.property_id = property.id
# INNER JOIN product_type on product_type.id = lot.product_type_id
# WHERE lot.company_id = {company_id}
# AND lot.product_type_id = {product_type}

# UNION ALL

# SELECT lopv.lot_id, lot.name as lot_name, 
# 	   property.name as property_name,
#        lopv.value as property_value,
#        lot.company_id,
#        lot.product_type_id,
#        product_type.name as product_name
# FROM lot_option_property_value lopv
# INNER JOIN lot on lot.id = lopv.lot_id
# INNER JOIN option_property on option_property.id = lopv.option_property_id
# INNER JOIN property on option_property.property_id = property.id
# INNER JOIN product_type on product_type.id = lot.product_type_id
# WHERE lot.company_id = {company_id}
# AND lot.product_type_id = {product_type}

# ORDER BY lot_id
# """

In [ ]:
db_conn = PFDBConnector()
info_df = db_conn.query(PF_KEY_PATH, PF_CA_CERT, PF_SQL_PASSWORD, BASE_QUERY).set_index("lot_id")
print("successful query")

label_filename = output_folder / f"label_{today_date}.csv"
info_df.to_csv(label_filename)

## spectra

In [10]:
BASE_QUERY = f"""
SELECT
	sp.id AS specimen_id,
	l.id AS lot_id,
	l.name AS lot_name,
	sp.date_scanned,
	sp.analyzer_id AS analyzer_id,
    l.company_id as company_id,
	p.id AS product_id,
    p.name AS product_name
FROM
	specimen sp
	INNER JOIN lot l ON l.id = sp.lot_id
	INNER JOIN product_type p on l.product_type_id = p.id
WHERE l.company_id = {company_id}
AND p.id = {product_type}
AND l.name NOT LIKE '%DELETED%'
ORDER BY l.id;
"""

In [11]:
SPECTRA_COLS = [
        "raw_data",
        "dark_ref_data",
        "white_ref_data",
        "dark_ref_scan_time",
        "white_ref_scan_time",
    ]
db_conn = PFDBConnector(
    sql_host="mysql8-uat.cgfknnbireqa.ap-southeast-1.rds.amazonaws.com", 
    sql_user="uat_user", 
    sql_dbname="platform_uat",
    ssh_host="54.179.47.210",
    key_type = 'RSA')
info_df = db_conn.query(PF_KEY_PATH, PF_CA_CERT, PF_SQL_PASSWORD, BASE_QUERY)
print("successful query")
efs_conn = BaseEFSConnector(url=PF_EFS_URL)
print(len(info_df))

# response = input(f'Found {info_df.shape[0]} rows. Proceed? [y]/n ')
# if response.lower() == 'n':
#     print('stopping data pull')
#     raise SystemExit()

spectra_df = efs_conn.fetch_spectra(info_df.specimen_id.values)
joined_df = info_df.merge(
    spectra_df.loc[:, SPECTRA_COLS], left_on="specimen_id", right_index=True
)
joined_df.set_index("lot_id", inplace=True)
print("Successfully queried from DB")
print(joined_df.head(n=10))
print(joined_df.shape)

spectra_filename = output_folder / f"spectra_{today_date}.csv"
joined_df.to_csv(spectra_filename)

successful query
240


Fetching from EFS: 100%|██████████| 5/5 [00:11<00:00,  2.38s/it]


Successfully queried from DB
        specimen_id  lot_name  date_scanned  analyzer_id  company_id   
lot_id                                                                 
59647        344215  C_30-M12    1779782643           96        1344  \
59647        344216  C_30-M12    1779782662           96        1344   
59647        344217  C_30-M12    1779782680           96        1344   
59647        344218  C_30-M12    1779782699           96        1344   
59647        344219  C_30-M12    1779782717           96        1344   
59647        344220  C_30-M12    1779782737           96        1344   
59647        344221  C_30-M12    1779782755           96        1344   
59647        344222  C_30-M12    1779782775           96        1344   
59647        344223  C_30-M12    1779782797           96        1344   
59647        344224  C_30-M12    1779782814           96        1344   

        product_id               product_name   
lot_id                                          
59647   

# by batch

In [10]:
batch = 5028
today_date = datetime.now().strftime("%y%m%d")
print (today_date)

251002


## label

In [ ]:
BASE_QUERY = f"""
WITH TargetLots AS (
    -- Identify the lots within the specified batches first to limit the dataset
    SELECT 
        l.id AS lot_id,
        l.name AS lot_name,
        l.company_id,
        l.product_type_id,
        pt.name AS product_name,
        lb.name AS batch_name
    FROM lot l
    INNER JOIN product_type pt ON l.product_type_id = pt.id
    INNER JOIN lot_batch_bridge br ON l.id = br.lot_id
    INNER JOIN lot_batch lb ON br.lot_batch_id = lb.id
    WHERE br.lot_batch_id IN ({batch})
),
PropertyValues AS (
    -- Combine numerical and option properties into one set
    SELECT 
        lnpv.lot_id, 
        p.name AS property_name, 
        lnpv.value AS property_value
    FROM lot_numerical_property_value lnpv
    INNER JOIN numerical_property np ON lnpv.numerical_property_id = np.id
    INNER JOIN property p ON np.property_id = p.id

    UNION ALL

    SELECT 
        lopv.lot_id, 
        p.name AS property_name, 
        lopv.value AS property_value
    FROM lot_option_property_value lopv
    INNER JOIN option_property op ON lopv.option_property_id = op.id
    INNER JOIN property p ON op.property_id = p.id
)
SELECT 
    tl.lot_id,
    tl.lot_name,
    pv.property_name,
    pv.property_value,
    tl.company_id,
    tl.product_type_id,
    tl.product_name,
    tl.batch_name
FROM TargetLots tl
INNER JOIN PropertyValues pv ON tl.lot_id = pv.lot_id
ORDER BY tl.lot_id;
"""

In [ ]:
db_conn = PFDBConnector()
info_df = db_conn.query(PF_KEY_PATH, PF_SQL_PASSWORD, BASE_QUERY).set_index("lot_id")
print("successful query")

info_filename = output_folder / f"info_batch{batch}_{today_date}.csv"
info_df.to_csv(info_filename)

successful query


## spectra

In [13]:
BASE_QUERY = f"""
SELECT
	sp.id AS specimen_id,
	l.id AS lot_id,
	l.name AS lot_name,
	sp.date_scanned,
	sp.analyzer_id AS analyzer_id
FROM
	specimen sp
	INNER JOIN lot l ON l.id = sp.lot_id
    INNER JOIN lot_batch_bridge br on br.lot_id = l.id
WHERE
	# l.company_id = 1243
    br.lot_batch_id IN ({batch})
ORDER BY
	l.id;
"""

In [ ]:
SPECTRA_COLS = [
        "raw_data",
        "dark_ref_data",
        "white_ref_data",
        "dark_ref_scan_time",
        "white_ref_scan_time",
    ]
db_conn = PFDBConnector()
info_df = db_conn.query(PF_KEY_PATH, PF_SQL_PASSWORD, BASE_QUERY)
print("successful query")
efs_conn = BaseEFSConnector(url=PF_EFS_URL)
print(len(info_df))

# response = input(f'Found {info_df.shape[0]} rows. Proceed? [y]/n ')
# if response.lower() == 'n':
#     print('stopping data pull')
#     raise SystemExit()

spectra_df = efs_conn.fetch_spectra(info_df.specimen_id.values)
joined_df = info_df.merge(
    spectra_df.loc[:, SPECTRA_COLS], left_on="specimen_id", right_index=True
)
joined_df.set_index("lot_id", inplace=True)
print("Successfully queried from DB")

spectra_filename = output_folder / f"spectra_batch{batch}_{today_date}.csv"
joined_df.to_csv(spectra_filename)

successful query
176


Fetching from EFS: 100%|██████████| 4/4 [00:15<00:00,  3.95s/it]


Successfully queried from DB
